# ML-09 — Validation and Research Claim Audit

This notebook audits the Week-5 decline/review model. The goal is to test whether the measured result survives an honest validation design.

**Public-safe language:** observed, measured, directional, decision-support.

**Data:** the 30,000-row anonymized starter CSV shipped with the repo. No warehouse scan is required, so this is designed to avoid the Colab RAM crash.

## 1. Two paper findings + my methodology questions

### Finding 1 — Click Capture by Position Tier

The FlyRank April 2026 report measures weighted CTR of **0.420% for positions 1–3**, **0.340% for positions 4–10**, and **0.050% for deep positions (50+)**. It describes this as a steep click drop-off. The report also states that this is a pattern study, not proof of cause and effect.

**Methodology question:** the outcome is an observed CTR aggregate, not a randomized outcome. I would ask whether query mix, brand/non-brand demand, device mix, and other differences between position tiers could explain part of the observed relationship. The constructive interpretation is: *higher visibility is associated with more click capture in this portfolio*. The stronger claim *moving a page upward will cause the same CTR lift* would need a stronger design or prospective test.

### Finding 2 — The Freshness Multiplier

The report measures a **5.43:1 growth-to-decline ratio for pages updated 31–90 days ago**. It separately reports that among 365+ day pages, the recently refreshed cohort had higher health and impressions than an older stale cohort. The report explicitly warns that the 361+ freshness bucket is small and unstable.

**Methodology question:** where does the refresh treatment come from, and what is the counterfactual? Pages are not randomly selected for refresh. Teams may refresh pages with stronger demand, strategic importance, or an existing upward trend. I would want the pre-refresh trend, a comparable control group, the treatment definition, and a time-separated post-refresh outcome before treating the measured difference as causal.

**Constructive takeaway:** these findings are useful measured portfolio patterns. The same standard should apply to my own model: define the label, keep prediction information strictly before the label, and validate on groups that represent the deployment question.

**Source:** FlyRank, *The State of AI-Driven SEO — April 2026*: https://state-of-seo-2026.flyrank.ai/

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PATH = Path('data/raw/content_refresh_anonymized.csv')
if not DATA_PATH.exists():
    import urllib.request
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/Aashusharma2005/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv',
        DATA_PATH
    )

df = pd.read_csv(DATA_PATH)
assert len(df) == 30000, f'Expected 30,000 starter rows, found {len(df):,}'
df['is_declining_label'] = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)
print(f'Rows: {len(df):,}')
print(f'Clients: {df.client_id.nunique():,}')
print(f'Declining label rate: {df.is_declining_label.mean():.3f}')

Rows: 30,000
Clients: 32
Declining label rate: 0.542


## 2. My model under an honest split (before/after)

The Week-5 reference model uses a Random Forest with balanced class weights. Here I compare the usual **random row holdout** with a **client-grouped holdout**. The grouped split is the more honest estimate for the question: *does this work for a client the model never saw during training?*

The score uses only baseline fields that are available before the decline label. `trend_direction` and `trend_pct` are excluded because the label is derived from them. Short-term trend windows are also excluded to avoid overlap with the target definition.

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

RANDOM_STATE = 42
NUMERIC = [
    'search_volume','competition','cpc','word_count','char_count',
    'impressions_90d','clicks_90d','sessions_90d','ai_sessions_90d',
    'days_with_impressions','days_with_sessions','content_age_days',
    'days_since_last_update','ctr','avg_position','engagement_rate',
    'scroll_rate','ai_traffic_pct'
]
CATEGORICAL = [
    'competition_level','content_type','main_intent','age_tier',
    'freshness_tier','word_count_tier','impression_tier','position_tier'
]
FORBIDDEN = {
    'trend_direction','trend_pct','is_declining_label',
    'impressions_last_30d','clicks_last_30d','sessions_last_30d',
    'impressions_prev_30d','clicks_prev_30d','sessions_prev_30d',
    'health_score','is_declining','is_underperformer','is_initial_refresh_candidate',
    'needs_ctr_fix','needs_engagement_fix','is_quick_win','needs_indexing','ai_opportunity'
}

for c in ['impressions_90d','clicks_90d','sessions_90d','ai_sessions_90d']:
    df[f'log_{c}'] = np.log1p(pd.to_numeric(df[c], errors='coerce').fillna(0))

MODEL_NUMERIC = [
    'search_volume','competition','cpc','word_count','char_count',
    'log_impressions_90d','log_clicks_90d','log_sessions_90d','log_ai_sessions_90d',
    'days_with_impressions','days_with_sessions','content_age_days',
    'days_since_last_update','ctr','avg_position','engagement_rate',
    'scroll_rate','ai_traffic_pct'
]

X_num = df[MODEL_NUMERIC].apply(pd.to_numeric, errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0)
X_cat = pd.get_dummies(df[CATEGORICAL].fillna('unknown').astype(str), prefix=CATEGORICAL, dtype=np.float32)
X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
y = df.is_declining_label.astype(int)

assert not (set(MODEL_NUMERIC) | set(CATEGORICAL)) & FORBIDDEN

def precision_at_50(y_true, score):
    order = np.argsort(-score)[:50]
    return float(np.asarray(y_true)[order].mean())

def run_split(train_idx, test_idx):
    model = RandomForestClassifier(
        n_estimators=150, max_depth=10, min_samples_leaf=25,
        class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=2
    )
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    p = model.predict_proba(X.iloc[test_idx])[:,1]
    yt = y.iloc[test_idx].to_numpy()
    pred = (p >= 0.5).astype(int)
    return {
        'accuracy': accuracy_score(yt,pred),
        'precision': precision_score(yt,pred,zero_division=0),
        'recall': recall_score(yt,pred,zero_division=0),
        'f1': f1_score(yt,pred,zero_division=0),
        'roc_auc': roc_auc_score(yt,p),
        'precision_at_50': precision_at_50(yt,p),
        'model': model, 'probabilities': p, 'test_indices': np.asarray(test_idx)
    }

all_idx = np.arange(len(df))
train_idx, test_idx = train_test_split(all_idx, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
random_result = run_split(train_idx, test_idx)

rng = np.random.default_rng(RANDOM_STATE)
clients = rng.permutation(df.client_id.drop_duplicates().to_numpy())
test_clients = set(clients[:max(1, int(round(0.20*len(clients))))])
mask = df.client_id.isin(test_clients).to_numpy()
group_result = run_split(all_idx[~mask], all_idx[mask])

comparison = pd.DataFrame([
    {'split':'Random stratified rows (before)', **{k:random_result[k] for k in ['accuracy','f1','roc_auc','precision_at_50']}},
    {'split':'Client holdout (after)', **{k:group_result[k] for k in ['accuracy','f1','roc_auc','precision_at_50']}}
])
print(comparison.to_string(index=False, float_format=lambda x:f'{x:.3f}'))
print(f'\nObserved Precision@50 gap (before - after): {random_result["precision_at_50"]-group_result["precision_at_50"]:.3f}')
print(f'Held-out clients: {len(test_clients)}')

                          split  accuracy    f1  roc_auc  precision_at_50
Random stratified rows (before)     0.690 0.719    0.758            0.900
         Client holdout (after)     0.674 0.641    0.752            0.740

Observed Precision@50 gap (before - after): 0.160
Held-out clients: 6


### Before/after interpretation

The grouped result is the safer number for claims about unseen clients. A random row split can share client-specific measurement patterns between train and test. The gap is therefore treated as a validation finding, not as proof of why performance changed.

For context, the repository README reports that the reference pipeline's learned model achieved roughly **0.68–0.74 Precision@50** on the bundled sample, versus roughly **0.24** for the hand-written baseline; exact values can vary by library version. This notebook recomputes its own metrics live rather than copying those numbers.

In [16]:
gap = random_result['precision_at_50'] - group_result['precision_at_50']
print(f'Random-split Precision@50: {random_result["precision_at_50"]:.3f}')
print(f'Client-holdout Precision@50: {group_result["precision_at_50"]:.3f}')
print(f'Observed gap: {gap:.3f}')
print('Decision-support boundary: use the client-holdout result when discussing generalization to unseen clients.')

Random-split Precision@50: 0.900
Client-holdout Precision@50: 0.740
Observed gap: 0.160
Decision-support boundary: use the client-holdout result when discussing generalization to unseen clients.


## 3. Leakage audit

### Label-derived leakage
`is_declining_label` is derived from `trend_direction`, which is computed from `trend_pct`. Neither is a model feature.

### Future / overlapping-window leakage
The decline target is based on short-term trend behavior, so `*_last_30d` and `*_prev_30d` fields are excluded from the model feature set. This is a conservative choice: a feature is legal only when it is clearly available before the prediction window.

### Decision-derived leakage
Existing action/health flags such as `is_declining`, `is_underperformer`, `is_initial_refresh_candidate`, `needs_ctr_fix`, and `needs_engagement_fix` are excluded. They may be useful as baselines, but using them as inputs could reproduce an existing decision rather than provide independent evidence.

### IDs
`content_id` and `client_id` are used only for identification/grouping, never as predictive features.

In [17]:
audit = pd.DataFrame({
    'category': ['Label-derived','Overlapping/future windows','Decision-derived flags','Identifiers'],
    'excluded': [
        'trend_direction, trend_pct, is_declining_label',
        'impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d',
        'health_score, is_declining, is_underperformer, is_initial_refresh_candidate, needs_ctr_fix, needs_engagement_fix, is_quick_win, needs_indexing, ai_opportunity',
        'content_id, client_id'
    ],
    'reason': [
        'Directly defines or reveals the target.',
        'Can overlap the target window; excluded for a conservative audit.',
        'May encode an existing product decision.',
        'Pseudonymous IDs are for grouping/joining, not prediction.'
    ]
})
print(audit.to_string(index=False))
print(f'\nFinal feature count: {X.shape[1]}')
print('Leakage audit: PASS — forbidden target/decision fields are absent from X.')

                  category                                                                                                                                                       excluded                                                            reason
             Label-derived                                                                                                                 trend_direction, trend_pct, is_declining_label                           Directly defines or reveals the target.
Overlapping/future windows                                             impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d Can overlap the target window; excluded for a conservative audit.
    Decision-derived flags health_score, is_declining, is_underperformer, is_initial_refresh_candidate, needs_ctr_fix, needs_engagement_fix, is_quick_win, needs_indexing, ai_opportunity                          May encode an existing product de

### Intentional leakage attack

To make the audit falsifiable, we deliberately add `trend_pct`, which is label-derived. A very strong score here is evidence that the test harness can detect a leak; it is **not** a model success.

In [18]:
from sklearn.tree import DecisionTreeClassifier
leaky = df.trend_pct.fillna(0).to_numpy().reshape(-1,1)
tr, te = train_test_split(all_idx, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
leaky_model = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
leaky_model.fit(leaky[tr], y.iloc[tr])
leaky_p = leaky_model.predict_proba(leaky[te])[:,1]
print(f'Intentionally leaky Precision@50: {precision_at_50(y.iloc[te].to_numpy(), leaky_p):.3f}')
print('Conclusion: trend_pct is label-derived and is excluded from the real model.')

Intentionally leaky Precision@50: 1.000
Conclusion: trend_pct is label-derived and is excluded from the real model.


## Real failure examples

These are false positives and false negatives from the **client-holdout** test. Only pseudonymous `content_id` is shown.

In [19]:
honest_idx = group_result['test_indices']
honest_p = group_result['probabilities']
honest_y = y.iloc[honest_idx].to_numpy()
errors = df.iloc[honest_idx][['content_id','impressions_90d','avg_position','days_since_last_update','word_count','ctr']].copy()
errors['predicted_probability'] = honest_p
errors['actual_declining'] = honest_y
errors['error_type'] = np.select([
    (errors.predicted_probability >= .5) & (errors.actual_declining == 0),
    (errors.predicted_probability < .5) & (errors.actual_declining == 1)
], ['false_positive','false_negative'], default='correct')
cols = ['content_id','predicted_probability','actual_declining','impressions_90d','avg_position','days_since_last_update','word_count','ctr']
print('False positives:')
print(errors[errors.error_type=='false_positive'].sort_values('predicted_probability',ascending=False).head(5)[cols].to_string(index=False, float_format=lambda x:f'{x:.3f}'))
print('\nFalse negatives:')
print(errors[errors.error_type=='false_negative'].sort_values('predicted_probability').head(5)[cols].to_string(index=False, float_format=lambda x:f'{x:.3f}'))
print('\nThese examples are counterexamples to the ranking rule, so the output remains decision-support for human review.')

False positives:
          content_id  predicted_probability  actual_declining  impressions_90d  avg_position  days_since_last_update  word_count   ctr
content_d2dffcc697a4                  0.744                 0             5091        14.100                      20    4496.000 0.200
content_331182ca4cae                  0.741                 0             3026        35.900                      20    3546.000 0.000
content_643f585dc7f7                  0.738                 0              761        25.100                      20    1980.000 0.390
content_f5013794ba57                  0.736                 0              881        15.700                      20    3622.000 0.000
content_00603b0349b4                  0.734                 0             1076        25.600                      20    2439.000 0.090

False negatives:
          content_id  predicted_probability  actual_declining  impressions_90d  avg_position  days_since_last_update  word_count    ctr
content_28b4223f4e5

## 4. Claim rewrite

**Too-strong claim:** “The model predicts which pages will decline and tells the content team what to refresh.”

**Evidence-safe rewrite:** “On this 30k-row anonymized starter dataset, the model produced a measured ranking of pages by probability of the observed decline-label proxy. The client-holdout split provides a more conservative directional measurement of performance on unseen clients than a random row split. The output is decision-support for human review; it does not establish causality, guarantee future traffic recovery, or predict Google's ranking algorithm.”

In [20]:
claim_rewrite = pd.DataFrame([
    {'old_claim':'The model predicts which pages will decline.','safe_claim':'The model ranks pages by measured probability of the observed decline proxy label on held-out data.'},
    {'old_claim':'Refresh these pages to recover traffic.','safe_claim':'Use the ranked output as decision-support for human review of pages that may merit investigation or refresh.'},
    {'old_claim':'The model generalizes to clients.','safe_claim':'The client-holdout result is a directional measurement on clients not used for training in this sample.'}
])
print(claim_rewrite.to_string(index=False))

                                   old_claim                                                                                                   safe_claim
The model predicts which pages will decline.          The model ranks pages by measured probability of the observed decline proxy label on held-out data.
     Refresh these pages to recover traffic. Use the ranked output as decision-support for human review of pages that may merit investigation or refresh.
           The model generalizes to clients.      The client-holdout result is a directional measurement on clients not used for training in this sample.


## Self-check

- [x] Two paper findings named and questioned constructively.
- [x] Label origin and validation design discussed.
- [x] Random vs client-grouped validation compared.
- [x] Leakage audit covers label-derived, overlapping-window, decision-derived, and ID fields.
- [x] Intentional leakage attack included.
- [x] Real false-positive and false-negative examples included.
- [x] Claims rewritten as observed, measured, directional, decision-support.
- [x] No private client names, URLs, or queries are used.

**Before submitting:** run **Runtime → Run all** in Colab, make sure every cell has output and no errors, then save/commit this executed notebook to `work/notebooks/w06_validation_audit.ipynb`.